<a href="https://colab.research.google.com/github/vedadapa/ml_foundations/blob/main/loan_deault_LinearRegression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Create a directory for Kaggle credentials if it doesn't exist
import os
os.makedirs('/root/.kaggle', exist_ok=True)

# Copy the Kaggle API key to the credentials directory
# This step assumes 'kaggle.json' is present in the current directory.
!cp kaggle.json /root/.kaggle/
# Set appropriate permissions for the Kaggle API key file
!chmod 600 /root/.kaggle/kaggle.json

cp: cannot stat 'kaggle.json': No such file or directory
chmod: cannot access '/root/.kaggle/kaggle.json': No such file or directory


In [2]:
# Download the specified dataset from Kaggle
!kaggle datasets download -d yasserh/loan-default-dataset

Dataset URL: https://www.kaggle.com/datasets/yasserh/loan-default-dataset
License(s): CC0-1.0
100% 4.89M/4.89M [00:01<00:00, 4.57MB/s]



In [3]:
# Unzip the downloaded dataset file
!unzip loan-default-dataset.zip

Archive:  loan-default-dataset.zip
  inflating: Loan_Default.csv        


In [4]:
# Import necessary libraries: pandas for data manipulation and numpy for numerical operations
import pandas as pd
import numpy as np

In [5]:
# Load the 'Loan_Default.csv' file into a pandas DataFrame
df = pd.read_csv('Loan_Default.csv')

In [6]:
# Display the first 5 rows of the DataFrame to get an initial look at the data
df.head()

,ID,year,loan_limit,Gender,approv_in_adv,loan_type,loan_purpose,Credit_Worthiness,open_credit,business_or_commercial,...,credit_type,Credit_Score,co-applicant_credit_type,age,submission_of_application,LTV,Region,Security_Type,Status,dtir1
0,24890,2019,cf,Sex Not Available,nopre,type1,p1,l1,nopc,nob/c,...,EXP,758,CIB,25-34,to_inst,98.728814,south,direct,1,45.0
1,24891,2019,cf,Male,nopre,type2,p1,l1,nopc,b/c,...,EQUI,552,EXP,55-64,to_inst,NaN,North,direct,1,NaN
2,24892,2019,cf,Male,pre,type1,p1,l1,nopc,nob/c,...,EXP,834,CIB,35-44,to_inst,80.019685,south,direct,0,46.0
3,24893,2019,cf,Male,nopre,type1,p4,l1,nopc,nob/c,...,EXP,587,CIB,45-54,not_inst,69.376900,North,direct,0,42.0
4,24894,2019,cf,Joint,pre,type1,p1,l1,nopc,nob/c,...,CRIF,602,EXP,25-34,not_inst,91.886544,North,direct,0,39.0


In [7]:
# Print the shape of the DataFrame (number of rows, number of columns)
df.shape

(148670, 34)

In [8]:
# Display the names of all columns in the DataFrame
df.columns

Index(['ID', 'year', 'loan_limit', 'Gender', 'approv_in_adv', 'loan_type',
       'loan_purpose', 'Credit_Worthiness', 'open_credit',
       'business_or_commercial', 'loan_amount', 'rate_of_interest',
       'Interest_rate_spread', 'Upfront_charges', 'term', 'Neg_ammortization',
       'interest_only', 'lump_sum_payment', 'property_value',
       'construction_type', 'occupancy_type', 'Secured_by', 'total_units',
       'income', 'credit_type', 'Credit_Score', 'co-applicant_credit_type',
       'age', 'submission_of_application', 'LTV', 'Region', 'Security_Type',
       'Status', 'dtir1'],
      dtype='object')

In [9]:
# Count the number of missing (NaN) values for each column in the DataFrame
df.isnull().sum()

,0
ID,0
year,0
loan_limit,3344
Gender,0
approv_in_adv,908
loan_type,0
loan_purpose,134
Credit_Worthiness,0
open_credit,0
business_or_commercial,0


In [10]:
# Display the distribution of values in the 'Status' column (the target variable)
df['Status'].value_counts()

,count
Status,
0,112031
1,36639


In [11]:
# Check for and count any duplicate rows in the DataFrame
df.duplicated().sum()

np.int64(0)

In [12]:
# Generate descriptive statistics for numerical columns in the DataFrame
df.describe()

,ID,year,loan_amount,rate_of_interest,Interest_rate_spread,Upfront_charges,term,property_value,income,Credit_Score,LTV,Status,dtir1
count,148670.000000,148670.0,1.486700e+05,112231.000000,112031.000000,109028.000000,148629.000000,1.335720e+05,139520.000000,148670.000000,133572.000000,148670.000000,124549.000000
mean,99224.500000,2019.0,3.311177e+05,4.045476,0.441656,3224.996127,335.136582,4.978935e+05,6957.338876,699.789103,72.746457,0.246445,37.732932
std,42917.476598,0.0,1.839093e+05,0.561391,0.513043,3251.121510,58.409084,3.599353e+05,6496.586382,115.875857,39.967603,0.430942,10.545435
min,24890.000000,2019.0,1.650000e+04,0.000000,-3.638000,0.000000,96.000000,8.000000e+03,0.000000,500.000000,0.967478,0.000000,5.000000
25%,62057.250000,2019.0,1.965000e+05,3.625000,0.076000,581.490000,360.000000,2.680000e+05,3720.000000,599.000000,60.474860,0.000000,31.000000
50%,99224.500000,2019.0,2.965000e+05,3.990000,0.390400,2596.450000,360.000000,4.180000e+05,5760.000000,699.000000,75.135870,0.000000,39.000000
75%,136391.750000,2019.0,4.365000e+05,4.375000,0.775400,4812.500000,360.000000,6.280000e+05,8520.000000,800.000000,86.184211,0.000000,45.000000
max,173559.000000,2019.0,3.576500e+06,8.000000,3.357000,60000.000000,360.000000,1.650800e+07,578580.000000,900.000000,7831.250000,1.000000,61.000000


In [13]:
# Calculate the correlation matrix for all numerical columns in the DataFrame
df.select_dtypes(include='number').corr()

,ID,year,loan_amount,rate_of_interest,Interest_rate_spread,Upfront_charges,term,property_value,income,Credit_Score,LTV,Status,dtir1
ID,1.000000,NaN,-0.000566,0.000442,0.002338,-0.005507,-0.004056,0.000990,0.002535,-0.001036,-0.005853,0.001703,-0.008132
year,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
loan_amount,-0.000566,NaN,1.000000,-0.150844,-0.377272,0.065556,0.174474,0.734249,0.456065,0.004438,0.038869,-0.036825,0.015029
rate_of_interest,0.000442,NaN,-0.150844,1.000000,0.614908,-0.076473,0.209330,-0.122613,-0.041809,-0.001331,-0.000220,0.022957,0.055124
Interest_rate_spread,0.002338,NaN,-0.377272,0.614908,1.000000,0.033037,-0.157139,-0.334571,-0.151333,-0.001738,0.040257,NaN,0.078178
Upfront_charges,-0.005507,NaN,0.065556,-0.076473,0.033037,1.000000,-0.054960,0.053043,0.016580,-0.001484,-0.031347,-0.019138,0.000115
term,-0.004056,NaN,0.174474,0.209330,-0.157139,-0.054960,1.000000,0.045117,-0.053785,-0.003149,0.106834,-0.000240,0.110572
property_value,0.000990,NaN,0.734249,-0.122613,-0.334571,0.053043,0.045117,1.000000,0.414883,0.002430,-0.215102,-0.048864,-0.056288
income,0.002535,NaN,0.456065,-0.041809,-0.151333,0.016580,-0.053785,0.414883,1.000000,0.000802,-0.066203,-0.065119,-0.267807
Credit_Score,-0.001036,NaN,0.004438,-0.001331,-0.001738,-0.001484,-0.003149,0.002430,0.000802,1.000000,-0.005533,0.004004,-0.000313


In [14]:
# Calculate the correlation of each numerical column with the 'Status' column and sort them in descending order
df.select_dtypes(include='number').corr()['Status'].sort_values(ascending=False)

,Status
Status,1.000000
dtir1,0.078083
LTV,0.038895
rate_of_interest,0.022957
Credit_Score,0.004004
ID,0.001703
term,-0.000240
Upfront_charges,-0.019138
loan_amount,-0.036825
property_value,-0.048864


In [30]:
# Define a list of columns to be dropped from the DataFrame
cols_to_drop = ['rate_of_interest', 'Interest_rate_spread',
                'Upfront_charges', 'dtir1']

# Drop the specified columns from the DataFrame. 'errors="ignore"' prevents errors if a column is not found.
df = df.drop(columns=cols_to_drop,errors="ignore")

In [23]:
# Fill missing values in 'income', 'property_value', and 'LTV' columns with their respective median values
df['income'] = df['income'].fillna(df['income'].median())
df['property_value'] = df['property_value'].fillna(df['property_value'].median())
df['LTV'] = df['LTV'].fillna(df['LTV'].median())

In [26]:
# Print the data type of the 'loan_limit' column
print(df['loan_limit'].dtype)
# Print the value counts for the 'loan_limit' column to understand its distribution
print(df['loan_limit'].value_counts())

object
loan_limit
cf     135348
ncf      9978
Name: count, dtype: int64


In [31]:
# Define a list containing the 'loan_limit' column name
cols_to_drop = ['loan_limit']
# Drop the 'loan_limit' column from the DataFrame
df = df.drop(columns=cols_to_drop,errors="ignore")

In [32]:
# Display the updated column names of the DataFrame after dropping some columns
df.columns

Index(['ID', 'year', 'Gender', 'approv_in_adv', 'loan_type', 'loan_purpose',
       'Credit_Worthiness', 'open_credit', 'business_or_commercial',
       'loan_amount', 'term', 'Neg_ammortization', 'interest_only',
       'lump_sum_payment', 'property_value', 'construction_type',
       'occupancy_type', 'Secured_by', 'total_units', 'income', 'credit_type',
       'Credit_Score', 'co-applicant_credit_type', 'age',
       'submission_of_application', 'LTV', 'Region', 'Security_Type',
       'Status'],
      dtype='object')

In [38]:
# Drop rows that have missing values in the 'term', 'age', or 'approv_in_adv' columns
df = df.dropna(subset=['term'])
df = df.dropna(subset=['age'])
df = df.dropna(subset=['approv_in_adv'])

In [40]:
# Verify that there are no longer any missing values in 'property_value', 'income', and 'LTV' columns
print(df[['property_value', 'income', 'LTV']].isnull().sum())

property_value    0
income            0
LTV               0
dtype: int64


In [41]:
# Recalculate and display the correlation matrix for all numerical columns after data cleaning
df.select_dtypes(include='number').corr()

,ID,year,loan_amount,term,property_value,income,Credit_Score,LTV,Status
ID,1.000000,NaN,-0.000500,-0.004057,NaN,NaN,-0.000940,NaN,0.001659
year,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
loan_amount,-0.000500,NaN,1.000000,0.174720,NaN,NaN,0.004736,NaN,-0.035340
term,-0.004057,NaN,0.174720,1.000000,NaN,NaN,-0.003061,NaN,-0.000255
property_value,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
income,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Credit_Score,-0.000940,NaN,0.004736,-0.003061,NaN,NaN,1.000000,NaN,0.003879
LTV,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Status,0.001659,NaN,-0.035340,-0.000255,NaN,NaN,0.003879,NaN,1.000000


In [46]:
# Define the feature columns (independent variables) for the model
features = ['loan_amount','term','property_value','income','Credit_Score','LTV']
# Define the target column (dependent variable) for the model
target = ['Status']

# Create the feature matrix X using the selected features
X= df[features]
# Create the target vector y using the target column
y = df[target]

In [49]:
# Import the train_test_split function for splitting data
from sklearn.model_selection import train_test_split

# Split the data into training and testing sets (80% train, 20% test)
# random_state ensures reproducibility of the split
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)
# Print the shapes of the resulting training and testing sets to verify their dimensions
print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)

(118016, 6)
(118016, 1)
(29505, 6)
(29505, 1)


In [51]:
# Import LinearRegression model from scikit-learn
from sklearn.linear_model import LinearRegression
# Import mean_squared_error for model evaluation
from sklearn.metrics import mean_squared_error
# Import numpy for numerical operations like square root
import numpy as np

# Initialize the Linear Regression model
model = LinearRegression()
# Train the model using the training data
model.fit(X_train,y_train)
# Make predictions on the test set
y_pred = model.predict(X_test)
# Calculate Mean Squared Error (MSE) between actual and predicted values
mse = mean_squared_error(y_test,y_pred)
# Calculate Root Mean Squared Error (RMSE)
rmse = np.sqrt(mse)
# Print the calculated MSE and RMSE, formatted to four decimal places
print(f"MSE: {mse:.4f}")
print(f"RMSE: {rmse:.4f}")
# Print the model's coefficients (weights) for each feature
print(f"Model weight (coefficients): {model.coef_}")
# Print the model's intercept (bias)
print(f"Model bias (intercept): {model.intercept_}")

MSE: 0.1838
RMSE: 0.4287
Model weight (coefficients): [[-8.56469824e-08  4.98708446e-05  0.00000000e+00  0.00000000e+00
   1.77888613e-05  8.34581894e-36]]
Model bias (intercept): [0.24506465]
